In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import year, sum, to_date
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
import os

In [2]:
# Create Spark Session

spark = SparkSession.builder \
    .appName("Retail_Spark_Project") \
    .getOrCreate()

print("Spark Session Created Successfully")


Spark Session Created Successfully


In [3]:
# Create Streaming Input Folder

os.makedirs("streaming_input", exist_ok=True)
os.makedirs("output", exist_ok=True)
os.makedirs("checkpoint", exist_ok=True)

In [4]:
# Defining Schema

stream_schema = StructType([
    StructField("ORDER_ID", StringType(), True),
    StructField("ORDER_DATE", StringType(), True),
    StructField("CUSTOMER_ID", StringType(), True),
    StructField("PRODUCT_ID", StringType(), True),
    StructField("CATEGORY", StringType(), True),
    StructField("SALES", DoubleType(), True)
])

In [5]:
# Read Streaming Data

stream_df = spark.readStream \
    .schema(stream_schema) \
    .option("header", True) \
    .csv("streaming_input")

# ORDER_DATE to proper Date
stream_df = stream_df.withColumn(
    "ORDER_DATE",
    to_date("ORDER_DATE", "yyyy-MM-dd")
)


In [6]:
# Aggregation (Yearly Sales)

stream_yearly = stream_df \
    .groupBy(year("ORDER_DATE").alias("YEAR")) \
    .agg(sum("SALES").alias("TOTAL_SALES"))

In [7]:
# foreachBatch (Correct for CSV Sink)

def write_yearly_batch(batch_df, epoch_id):
    batch_df.write \
        .mode("overwrite") \
        .option("header", True) \
        .csv("output/streamed_yearly_sales")

query = stream_yearly.writeStream \
    .outputMode("complete") \
    .foreachBatch(write_yearly_batch) \
    .option("checkpointLocation", "checkpoint") \
    .start()

print("Streaming Started...")

Streaming Started...


In [9]:
# 7Simulate Streaming Input

spark_df = spark.read.csv(
    "retail_sales_cleaned.csv",
    header=True,
    inferSchema=True
)

In [10]:
# Adding small batch to streaming folder
spark_df.limit(100).write \
    .option("header", True) \
    .mode("append") \
    .csv("streaming_input")

print("Added 100 rows to streaming input.")

Added 100 rows to streaming input.


In [12]:
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType

result_schema = StructType([
    StructField("YEAR", IntegerType(), True),
    StructField("TOTAL_SALES", DoubleType(), True)
])

spark.read.csv(
    "output/streamed_yearly_sales",
    header=True,
    schema=result_schema
).show()

+----+------------------+
|YEAR|       TOTAL_SALES|
+----+------------------+
|2016|25027.499999999996|
+----+------------------+



In [11]:
# Monitor Streaming

print("Streaming Query Active:", query.isActive)
print("Streaming Query Status:", query.status)

Streaming Query Active: True
Streaming Query Status: {'message': 'Processing new data', 'isDataAvailable': True, 'isTriggerActive': True}
